In [2]:
import geopandas as gpd

In [3]:
status = gpd.read_file(r'C:\Users\dav\Documents\EcoIndex\_GIS_CORE\_DAIRYNZ\base_layers\eco-index-current-status-and-restoration-priority-for-nz.gpkg')

In [11]:
status.to_file('../BaseLayersEco-index/Eco-status_old.gpkg')


# existing_areas_aggregation

In [61]:
import re

def dollar_to_decimal(value):
    if isinstance(value, float):
        return value  # If the value is already a float, return it as is.
    if isinstance(value, str):
        if value == "No data":
            return value
        # Clean the string using a regular expression to remove non-numeric, non-decimal, and non-minus characters.
        cleaned_str = re.sub(r"[^\d.-]", "", value.strip())
        # Check if the cleaned string is a valid decimal number
        if re.match(r"^-?\d+(\.\d+)?$", cleaned_str):
            try:
                decimal_value = float(cleaned_str)
                return decimal_value
            except ValueError:
                raise ValueError(
                    f"Invalid dollar figure after cleaning: {cleaned_str} from original {value}"
                )

    return None  # Return None for invalid inputs, empty strings, or non-string/non-float data
    
#### join to costs
costs_look_up_ena = pd.read_csv("../BaseLayersEco-index/CostsLookUpTable_ENA_v110924.csv", skiprows=[0], header=0)

# costs_look_up_ena['Calculated fence maintenance cost_m\n']#[['BASE FENCE MAINTENANCE']]
for i in costs_look_up_ena.columns:
    print(i)

cost_cols_ena_renaming = {
'LCDB_5 Land cover class': 'LCDB class',
'Wetland contex': 'Wetland context',
# 'Existing Natural Area': 
'ENA Group': 'ENA Group',
# 'non-native animal species management need'
# 'non-native plant species management need': 
# 'BASE ONGOING non-native animal species MAINTENANCE': 
'Calculated non-native animal species cost': 'Phase3: Non-native animal species cost p.a. (per ha)',
# 'BASE ONGOING non-native plant species MAINTENANCE': 
'Calculated non-native plant species cost': 'Phase3: Non-native plant species cost p.a. (per ha)',
# 'Fencing likihood': 
# 'BASE FENCE COST': 
# 'Calculated fence cost': 
'Calculated fence maintenance cost_m\n': "Phase3: Fence Maintenance costs p.a. (per metre)",
}
str_cols = ['LCDB_5 Land cover class', 'Wetland contex', 'ENA Group']
decimal_cols_ena = [i for i in cost_cols_ena_renaming.keys() if i not in str_cols]

for col in decimal_cols_ena:
    print(col)
    costs_look_up_ena[col] = costs_look_up_ena[col].apply(dollar_to_decimal)

costs_look_up_ena = costs_look_up_ena[cost_cols_ena_renaming.keys()]
costs_look_up_ena = costs_look_up_ena.rename(cost_cols_ena_renaming, axis=1)

anys = costs_look_up_ena.loc[costs_look_up_ena['Wetland context'] == 'any', :].copy()
anys['Wetland context'] = anys.shape[0]*[['Yes', 'No']]
anys['Wetland context'] = anys.shape[0]*[['Yes', 'No']]

costs_look_up_ena['LCDB class'] = costs_look_up_ena['LCDB class'].str.replace('–', '-')
costs_look_up_ena = pd.concat([
    anys.explode('Wetland context'), # the ones we have added
    costs_look_up_ena[costs_look_up_ena['Wetland context'] != 'any']
]).reset_index(drop=True)
costs_look_up_ena['Wetland context'] = costs_look_up_ena['Wetland context'].str.lower()


LCDB_5 Land cover class
Wetland contex
Existing Natural Area
ENA Group
non-native animal species management need
non-native plant species management need
BASE ONGOING non-native animal species MAINTENANCE
Calculated non-native animal species cost
BASE ONGOING non-native plant species MAINTENANCE
Calculated non-native plant species cost
Fencing likihood
BASE FENCE COST_m
Calculated fence cost_m
BASE FENCE MAINTENANCE
Calculated fence maintenance cost_m

Unnamed: 15
Unnamed: 16
Calculated non-native animal species cost
Calculated non-native plant species cost
Calculated fence maintenance cost_m



In [71]:
!dir '../BaseLayerEco-index/'

Parameter format not correct - "BaseLayerEco-index".


In [72]:
costs_look_up_ena[['LCDB class', 'Wetland context', 'ENA Group']].to_csv('../BaseLayersEco-index/LCDB_to_ENA_Group.csv')


In [ ]:

#### join
ena_cost = mature_regen.merge(costs_look_up_ena, left_on=['LCDBLandCoverClass', 'LCDBWetlandContext'],right_on=['LCDB class', 'Wetland context'], how='inner')

# should be none
# assert ena_cost.shape[0] == mature_regen.shape[0] # It appears Estuarine Open Water with Wetland Context = No is not in the costs look up table (it is the only one missing for all NZ). I don't think this is an issue but may be worth fixing for future work
assert ena_cost[ena_cost['LCDB class'].isna()][['LCDBLandCoverClass', 'LCDBWetlandContext']].drop_duplicates().shape[0] == 0

ena_cost = ena_cost.drop(['LCDB class','Wetland context', 'ENA Group', 'Realm',
       'ExistingNaturalArea',], axis=1)


# RTZ

In [3]:
%%time
trz_path = r'C:\Users\dav\Documents\EcoIndex\Eco-index Master GIS Layers (1)\Ei_TargetRestorationZone\Ei_TRZ_NZ_190623.shp'
trz = gpd.read_file(trz_path)
trz.sindex
trz.to_file('../BaseLayersEco-index/Eco-index_RestorationTargetZone_v190623.gpkg')

CPU times: total: 1.36 s
Wall time: 1.4 s


# Catchments

In [46]:
catch = gpd.read_file(r'C:\Users\dav\Documents\EcoIndex\Eco-index Master GIS Layers (1)\Ei_Catchments\Ei_Catchments_v080623_NZTM.shp')
# catch.to_file('../BaseLayersEco-index/Eco-index_Catchments_v080623.gpkg')

# Ecosystem Projector

In [44]:
%%time

import pandas as pd
proj = gpd.read_file(r'C:\Users\dav\Documents\EcoIndex\Eco-index Master GIS Layers (1)\PublicVersions\Pre-human Native Ecosystem Locations 101123.shp')
proj = proj.rename({'PNVWmacron':'ExpectedEcosystemType'}, axis=1)
proj['ExpectedEcosystemType'] = proj['ExpectedEcosystemType'].str.replace('totara', 'tōtara').str.replace('kamahi', 'kāmahi')
proj['ExpectedEcosystemType'] = proj['ExpectedEcosystemType'].str.replace('Wetland_Unclassified', 'Wetland Unclassified')
proj['ExpectedEcosystemType'] = proj['ExpectedEcosystemType'].str.replace(
    "Hall's tōtara-miro/kāmahi-southern rata broadleaf forest", 
    "Hall's tōtara-miro/kāmahi-southern rata-broadleaf forest"
)



tot_pnvw = pd.read_csv(
    "../BaseLayersEco-index/TableOfTruth_PNVW_Mapping.csv"
)
tot_pnvw = tot_pnvw[
    [
        "Potential Natural Vegetation and Wetland Type",
        "Eco_index Realm (not same as Costanza or IUCN)",
          "Aggregated Ecosystem type for costings calculations",
          "ECOSYSTEM VALUATION BIOME USE THIS FOR Ecosystem Services Valuer"
    ]
].copy()

tot_pnvw = tot_pnvw.rename(
    {
        "Potential Natural Vegetation and Wetland Type": "PNVWmacron",
        "Eco_index Realm (not same as Costanza or IUCN)": "Realm",
          "Aggregated Ecosystem type for costings calculations": "AggregatedEcosystemType",
          "ECOSYSTEM VALUATION BIOME USE THIS FOR Ecosystem Services Valuer":"EcosystemValuationBiome"
    },
    axis=1,
)
tot_pnvw['PNVWmacron'] = tot_pnvw['PNVWmacron'].str.replace('totara', 'tōtara').str.replace('kamahi', 'kāmahi')
tot_pnvw['PNVWmacron'] = tot_pnvw['PNVWmacron'].str.replace('Wetland_Unclassified', 'Wetland Unclassified')

tot_pnvw = pd.concat([tot_pnvw, 
    pd.DataFrame(
        {
            "PNVWmacron": ["unclassified"],
                # "unclassified wetland",
                # "Hall's tōtara-miro/kāmahi-southern rata broadleaf forest",
#                 # "Hall's totara-miro/kamahi-southern rata broadleaf forest",
#                 "Hall's tōtara-miro-rimu/kāmahi-southern rata-broadleaf forest"
            "Realm": ["unclassified"],# "Freshwater Wetland",],#, "Terrestrial", "Terrestrial"],
            "AggregatedEcosystemType": [None],#'Wetland-fertile', ],#"Mid-altitude forest", "Mid-altitude forest"]
            "EcosystemValuationBiome": [None],#'Wetlands', ],#"Temperate Forest", "Temperate Forest"]
        }
    )]
)
proj = proj.merge(tot_pnvw, left_on='ExpectedEcosystemType', right_on='PNVWmacron', how='left')
proj.sindex

# # make sure we have mappings for all EcosystemTypes:
assert proj[proj.Realm.isna()].ExpectedEcosystemType.unique().shape[0] == 0

proj[['ExpectedEcosystemType', 'Realm', 'geometry']].to_file('../BaseLayersEco-index/Eco-index_EcosystemProjector_v260924.gpkg')
proj[['ExpectedEcosystemType', 'AggregatedEcosystemType', 'Realm', 'EcosystemValuationBiome', 'geometry']].to_file('../BaseLayersEco-index/Eco-index_EcosystemProjector_Details_v260924.gpkg')

CPU times: total: 8.55 s
Wall time: 8.73 s


In [48]:
proj = proj[['ExpectedEcosystemType', 'AggregatedEcosystemType', 'Realm', 'EcosystemValuationBiome', 'geometry']].copy()

In [32]:
[i for i in tot_pnvw['PNVWmacron'] if i.startswith("Hall's tōtara-miro/kāmahi-southern rata broadleaf forest"[:39])]

["Hall's tōtara-miro/kāmahi-southern rata-broadleaf forest"]

In [74]:
proj2[['ExpectedEcosystemType', 'AggregatedEcosystemType']].to_csv('../BaseLayersEco-index/ExpectedEcosystemType_to_AggregatedEcosystemType.csv', index=False)

In [7]:
# proj = gpd.read_file('../BaseLayersEco-index/Eco-index_EcosystemProjector_Details_v260924.gpkg')
proj[['ExpectedEcosystemType', 'AggregatedEcosystemType', 'geometry']].to_file('../BaseLayersEco-index/Eco-index_EcosystemProjector_Public_v260924.gpkg')

## Ecosystem Projector by catchment

In [49]:
%%time
catch.sindex
proj.sindex
proj_catch = catch.overlay(proj)

CPU times: total: 2min 29s
Wall time: 2min 30s


C:\Users\dav\miniconda3_9\envs\eco\Lib\site-packages\geopandas\geodataframe.py:2675: UserWarning: `keep_geom_type=True` in overlay resulted in 3248 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  return geopandas.overlay(


In [50]:
proj_catch[['Catchment', 'ExpectedEcosystemType', 'AggregatedEcosystemType', 'Realm', 'EcosystemValuationBiome', 'geometry']].to_file('../BaseLayersEco-index/Eco-index_EcosystemProjector_Details__Catchments_v290824.gpkg')

## Ecosystem Projector by catchment Restorable

In [69]:
%%time
restorable = gpd.read_file('../BaseLayersEco-index/Eco-index_RestorableAreas_v290824.gpkg')
proj_catch_restorable = proj_catch.overlay(restorable)
proj_catch_restorable

C:\Users\dav\miniconda3_9\envs\eco\Lib\site-packages\geopandas\geodataframe.py:2675: UserWarning: `keep_geom_type=True` in overlay resulted in 28260 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  return geopandas.overlay(


In [75]:
proj_catch_restorable[['Catchment', 'EcosystemType', 'Realm', 'geometry']].to_file('../BaseLayersEco-index/Eco-index_RestorableAreas__Realms_v290824.gpkg')

## Ecosystem Projector Realms only

In [67]:
proj_catch.sindex
realms = proj_catch[['Catchment', 'Realm', 'geometry']].dissolve(['Catchment', 'Realm'])
realms = realms.reset_index().explode().reset_index(drop=True)
realms[['Catchment', 'Realm', 'geometry']].to_file('../BaseLayersEco-index/Eco-index_EcosystemProjector__RealmsOnly_v290824.gpkg')

# Land Cover Snapshot

In [19]:
lcdb_path = r'../BaseLayersExternal/lcdb-v5/lcdb-v50-land-cover-database-version-50-mainland-new-zealand.shp'
lcdb = gpd.read_file(lcdb_path)
lcdb = lcdb.to_crs('epsg:2193')
lcdb = lcdb[['Name_2018', 'Wetland_18', 'Onshore_18', 'geometry']].copy()
lcdb.sindex

In [20]:
import pandas as pd
tot = pd.read_csv('../BaseLayersEco-index/TableOfTruth_LCDB_Mapping.csv')
tot.loc[tot['LCDB_5 Land cover class'] == 'Forest – Harvested', 'LCDB_5 Land cover class'] = 'Forest - Harvested'
tot['LCDB_5 Land cover class'] = tot['LCDB_5 Land cover class'].str.replace('–', '-')
tot['Land cover status'] = tot['Land cover status'].str.strip()
tot
anys = tot.loc[tot['LCDB Wetland context'] == 'any', :].copy()
anys['LCDB Wetland context'] = anys.shape[0]*[['Yes', 'No']]
anys['LCDB Wetland context'] = anys.shape[0]*[['Yes', 'No']]

tot = pd.concat([
    anys.explode('LCDB Wetland context'), # the ones we have added
    tot[tot['LCDB Wetland context'] != 'any']
]).reset_index(drop=True)

# tot['Land cover status'] = tot['Land cover status']
tot['LCDB Wetland context'] = tot['LCDB Wetland context'].str.lower()
tot['EcosystemValuationBiome'] = tot['ECOSYSTEM VALUATION BIOME USE THIS FOR Ecosystem Services Valuer'].copy()
tot_sub = tot[['LCDB_5 Land cover class', 'LCDB Wetland context', 'Land cover status', 'Eco-Index Realm (Generalised, not IUCN)', 'Existing Natural Area', 'EcosystemValuationBiome']]
# display(tot_sub.loc[tot_sub['LCDB_5 Land cover class'].str.startswith('Forest')])
tot_sub

,LCDB_5 Land cover class,LCDB Wetland context,Land cover status,"Eco-Index Realm (Generalised, not IUCN)",Existing Natural Area,EcosystemValuationBiome
0,Alpine Grass/Herbfield,yes,Mature,Terrestrial,Existing Natural Area,Shrubland and shrubby woodland
1,Alpine Grass/Herbfield,no,Mature,Terrestrial,Existing Natural Area,Shrubland and shrubby woodland
2,Broadleaved Indigenous Hardwoods,yes,Regenerating,Terrestrial,Existing Natural Area,Temperate Forest
3,Broadleaved Indigenous Hardwoods,no,Regenerating,Terrestrial,Existing Natural Area,Temperate Forest
4,Herbaceous Freshwater Vegetation,yes,Mature,Freshwater,Existing Natural Area,Wetlands
...,...,...,...,...,...,...
63,Matagouri or Grey Scrub,no,Regenerating,Terrestrial,Existing Natural Area,Shrubland and shrubby woodland
64,Sub Alpine Shrubland,yes,Mature,Freshwater,Existing Natural Area,Wetlands
65,Sub Alpine Shrubland,no,Regenerating,Terrestrial,Existing Natural Area,Shrubland and shrubby woodland
66,Deciduous Hardwoods,yes,Mature,Freshwater,Existing Natural Area,Wetlands


In [39]:
print(lcdb.shape)
lcdb_tot = lcdb.merge(tot_sub, how='left', left_on = ['Name_2018', 'Wetland_18'], right_on=['LCDB_5 Land cover class', 'LCDB Wetland context'])
print(lcdb_tot.shape)
lcdb_tot[lcdb_tot['LCDB_5 Land cover class'].isna()]['Name_2018'].unique()

(511104, 4)
(511104, 10)


array(['Not land'], dtype=object)

In [40]:
lcdb_tot = lcdb.merge(tot_sub, how='inner', left_on = ['Name_2018', 'Wetland_18'], right_on=['LCDB_5 Land cover class', 'LCDB Wetland context'])

In [41]:
lcdb_tot['LandCoverStatus'] = lcdb_tot['Land cover status']
lcdb_tot['LandCoverStatus'] = lcdb_tot['LandCoverStatus'].str.strip()

new_land_status_mapping = {'Mature':'Mature Ecosystems', 
                           'Potentially available for reconstruction':'Reconstruction Opportunity',
                           'Regenerating':'Regenerating Ecosystems', 
                           'Unavailable for reconstruction':'Built-up and Transport Areas'}

lcdb_tot['LandCoverStatus'] = lcdb_tot.LandCoverStatus.map(new_land_status_mapping)

In [42]:
lcdb_tot = lcdb_tot[['LCDB_5 Land cover class', 'LCDB Wetland context', 'LandCoverStatus',
       'Eco-Index Realm (Generalised, not IUCN)', 'EcosystemValuationBiome', 'Existing Natural Area', 'geometry']]
lcdb_tot.columns
lcdb_tot = lcdb_tot.rename({'LCDB_5 Land cover class':'LCDBLandCoverClass',
                            'LCDB Wetland context':'LCDBWetlandContext',
                            'Eco-Index Realm (Generalised, not IUCN)': 'Realm',
                            'Existing Natural Area':'ExistingNaturalArea'}, axis=1)
    

In [43]:
lcdb_tot

,LCDBLandCoverClass,LCDBWetlandContext,LandCoverStatus,Realm,EcosystemValuationBiome,ExistingNaturalArea,geometry
0,Herbaceous Saline Vegetation,yes,Mature Ecosystems,Saline,Coastal systems,Existing Natural Area,"POLYGON ((1613722.435 5425797.372, 1613723.153..."
1,Herbaceous Saline Vegetation,yes,Mature Ecosystems,Saline,Coastal systems,Existing Natural Area,"POLYGON ((1816770.219 5947804.627, 1816778.549..."
2,Mangrove,yes,Mature Ecosystems,Saline,Mangroves,Existing Natural Area,"POLYGON ((1715672.186 5958842.706, 1715665.266..."
3,Herbaceous Saline Vegetation,yes,Mature Ecosystems,Saline,Coastal systems,Existing Natural Area,"POLYGON ((1705330.918 6088979.74, 1705316.037 ..."
4,Estuarine Open Water,no,Mature Ecosystems,Saline,Coastal systems,Existing Natural Area,"POLYGON ((1761684.636 5789742.527, 1761680.213..."
...,...,...,...,...,...,...,...
511085,Low Producing Grassland,no,Reconstruction Opportunity,Modified,NaN,NaN,"POLYGON ((1785112.194 5684560.595, 1785108.792..."
511086,Indigenous Forest,no,Mature Ecosystems,Terrestrial,Temperate Forest,Existing Natural Area,"POLYGON ((1607510.75 5432591.699, 1607542.075 ..."
511087,Exotic Forest,no,Reconstruction Opportunity,Modified,NaN,NaN,"POLYGON ((1603592.31 5269947.382, 1603608.224 ..."
511088,Herbaceous Freshwater Vegetation,yes,Mature Ecosystems,Freshwater,Wetlands,Existing Natural Area,"POLYGON ((1822629.596 5477414.336, 1822617.531..."


In [44]:
lcdb_tot.to_file('../BaseLayersEco-index/Eco-index_LandCoverSnapshot_Details_v290924.gpkg')

## Land Cover Snapshot By Catchment

In [47]:
%%time
lcdb_tot.sindex, catch.sindex
snap_catch = lcdb_tot.overlay(catch[['Catchment', 'geometry']])

CPU times: total: 36min 57s
Wall time: 37min


C:\Users\dav\miniconda3_9\envs\eco\Lib\site-packages\geopandas\geodataframe.py:2675: UserWarning: `keep_geom_type=True` in overlay resulted in 282 dropped geometries of different geometry types than df1 has. Set `keep_geom_type=False` to retain all geometries
  return geopandas.overlay(


In [48]:
snap_catch.to_file('../BaseLayersEco-index/Eco-index_LandCoverSnapshot_Details__Catchments_v290924.gpkg')

In [9]:
# snap_catch = gpd.read_file('../BaseLayersEco-index/Eco-index_LandCoverSnapshot__Catchments_v290924.gpkg')
snap_catch = snap_catch.drop(['Realm', 'ExistingNaturalArea'], axis=1)
snap_catch = snap_catch.dissolve(['LCDBLandCoverClass', 'LCDBWetlandContext', 'LandCoverStatus', 'Catchment']).explode()
snap_catch.head()

geometry
LCDBLandCoverClass     LCDBWetlandContext LandCoverStatus Catchment                                                   
Alpine Grass/Herbfield no                 Mature          Aparima    POLYGON ((1199465.048 4924720.239, 1199534.408...
                                                          Aparima    POLYGON ((1200309.744 4924766.983, 1200289.8 4...
                                                          Aparima    POLYGON ((1201376.997 4927974.921, 1201400.036...
                                                          Aparima    POLYGON ((1204500.508 4928418.259, 1204480.168...
                                                          Aparima    POLYGON ((1200371.384 4928544.545, 1200396.956...

In [10]:
snap_catch = snap_catch.reset_index()
snap_catch.to_file('../BaseLayersEco-index/Eco-index_LandCoverSnapshot__Public_v290824.gpkg')

# File shuffling:

In [18]:
snap_catch

,LCDBLandCoverClass,LCDBWetlandContext,LandCoverStatus,Catchment,geometry
0,Alpine Grass/Herbfield,no,Mature,Aparima,"POLYGON ((1199465.048 4924720.239, 1199534.408..."
1,Alpine Grass/Herbfield,no,Mature,Aparima,"POLYGON ((1200309.744 4924766.983, 1200289.8 4..."
2,Alpine Grass/Herbfield,no,Mature,Aparima,"POLYGON ((1201376.997 4927974.921, 1201400.036..."
3,Alpine Grass/Herbfield,no,Mature,Aparima,"POLYGON ((1204500.508 4928418.259, 1204480.168..."
4,Alpine Grass/Herbfield,no,Mature,Aparima,"POLYGON ((1200371.384 4928544.545, 1200396.956..."
...,...,...,...,...,...
470986,Urban Parkland/Open Space,no,Potentially available for reconstruction,Ōtaki,"POLYGON ((1792221.243 5501105.517, 1792207.791..."
470987,Urban Parkland/Open Space,no,Potentially available for reconstruction,Ōtaki,"POLYGON ((1791400.474 5501956.629, 1791393.583..."
470988,Urban Parkland/Open Space,no,Potentially available for reconstruction,Ōtaki,"POLYGON ((1789828.442 5503720.342, 1789840.212..."
470989,Urban Parkland/Open Space,no,Potentially available for reconstruction,Ōtaki,"POLYGON ((1786367.575 5508595.86, 1786355.393 ..."


In [13]:
snap_catch_simple_2m = snap_catch.copy()
snap_catch_simple_2m['geometry'] = snap_catch_simple_2m.simplify(tolerance=2)
# snap_catch_simple_2m.to_file('../BaseLayersEco-index/Eco-index_LandCoverSnapshot__Public_Simple_v290824.gpkg')


In [14]:
snap_catch_simple_2m.to_file('../BaseLayersEco-index/Eco-index_LandCoverSnapshot__Public_Simple_v290824.gpkg')


In [90]:

import shutil
from pathlib import Path

# Define the base directory where the gpkg files should be copied
base_dir = Path('../OutputArtifacts')  # Replace with your actual path

files_to_copy = {}

# Iterate over all subdirectories (one level deep)
for subdir in base_dir.iterdir():
    if subdir.is_dir():  # Check if it's a directory
        # Find all .gpkg files in the subdirectory
        for gpkg_file in subdir.glob('*.gpkg'):
            # Construct the destination path
            if 'B01' in str(gpkg_file) or 'B02' in str(gpkg_file): 
                continue
            if gpkg_file.name.startswith('A'):
                destination = base_dir / gpkg_file.name
                files_to_copy[gpkg_file] = destination
                
                # Copy the file to the base directory
                shutil.copy(gpkg_file, destination)
                print(f'Copied {gpkg_file} to {destination}')

Copied ..\OutputArtifacts\A01_ThreatenedEnvironment\A01_Threatened_Environment_20240624.gpkg to ..\OutputArtifacts\A01_Threatened_Environment_20240624.gpkg
Copied ..\OutputArtifacts\A02_ConnectivitySteppingStones\A02_Connectivity_20240916.gpkg to ..\OutputArtifacts\A02_Connectivity_20240916.gpkg
Copied ..\OutputArtifacts\A03_NativeVegetationProximity\A03_NativeVegetationProximity_20240829.gpkg to ..\OutputArtifacts\A03_NativeVegetationProximity_20240829.gpkg
Copied ..\OutputArtifacts\A04_LegalProtection\A04_LegalProtection_20240829.gpkg to ..\OutputArtifacts\A04_LegalProtection_20240829.gpkg
Copied ..\OutputArtifacts\A05_RelativeAffordability\A05_RelativeAfforability_20240829.gpkg to ..\OutputArtifacts\A05_RelativeAfforability_20240829.gpkg
Copied ..\OutputArtifacts\A06_RiparianBenefit\A06_RiparianBenefit_20240829.gpkg to ..\OutputArtifacts\A06_RiparianBenefit_20240829.gpkg
Copied ..\OutputArtifacts\A07_NativeVegetationShapeImprovement\A07_NativeVegetationShapeImprovement_20240906.gpkg